# Customer Churn Prediction & Business Insight System

### Zeravia Artificial Intelligence Training & Internship Program

**Project Type:** Machine Learning Classification & Regression  
**Primary Model:** Logistic Regression  
**Secondary Model:** Linear Regression  
**Dataset:** Telco Customer Churn Dataset

---

## Project Overview

This project develops an end-to-end machine learning solution for predicting customer churn and generating business insights.

The analysis follows the complete machine learning workflow:

**Business Problem → Data Exploration → Data Cleaning → Exploratory Data Analysis → Data Visualisation → Statistical Analysis → Data Preprocessing → Train-Test Split → Logistic Regression → Model Evaluation → Feature Interpretation → Customer Risk Insights → Linear Regression → Business Recommendations**

### Objective

The primary objective is to identify customers who are likely to churn and understand the customer characteristics associated with higher or lower churn risk.

The project also includes a Linear Regression extension to demonstrate the difference between classification and regression problems.

# 1. Understand the Business Problem

## 1.1 Business Objective

The client is a subscription-based company that wants to understand customer behaviour and identify customers who may stop using its services.

Customer churn refers to a customer discontinuing or leaving the company's service. In this project, the `Churn` variable will be used as the target variable for predicting whether a customer is likely to churn.

The goal is to:

- Analyse customer behaviour and identify patterns associated with churn.
- Build a machine learning model to classify customers according to their churn outcome.
- Evaluate the performance of the churn prediction model.
- Identify important customer characteristics associated with churn.
- Group customers into practical risk categories.
- Provide actionable recommendations that may help the client reduce customer churn.

## 1.2 Target Variable

The target variable is:

**`Churn`**

It represents whether a customer has left the service.

- **Yes** → Customer churned
- **No** → Customer did not churn

Since the target contains two possible outcomes, this is a **binary classification problem**.

## 1.3 Business Impact

Accurately identifying customers who are more likely to churn can help the company prioritise retention efforts.

For example, the company could focus customer support, retention offers, or engagement campaigns on customers identified as having higher churn risk.

The model's predictions should be treated as risk indicators rather than proof that a particular customer will leave.

# 2. Python Environment & Project Setup

This project is developed using Python in a Jupyter Notebook environment within GitHub Codespaces.

The main libraries used are:

- **Pandas** – data loading, manipulation and analysis
- **NumPy** – numerical operations
- **Matplotlib** – data visualisation
- **Seaborn** – statistical visualisation
- **Scikit-learn** – machine learning models, preprocessing and evaluation

The project follows a clean and organised notebook structure with readable Python code.

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# 3. Data Loading & Exploration

The Telco Customer Churn dataset contains historical customer information including demographic details, service subscriptions, contract information, billing information, tenure, and customer churn status.

The dataset will be loaded using Pandas and inspected to understand its structure, data types, dimensions, and basic statistical characteristics.

In [ ]:
# Load the dataset

df = pd.read_csv("data/telco_data.csv")

print("Dataset loaded successfully!")

In [ ]:
# Display the first five rows

df.head()

In [ ]:
# Check the number of rows and columns

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Display information about columns and data types

df.info()

In [ ]:
# Display descriptive statistics for numerical columns

df.describe()

## 3.1 Missing Value Overview

Before performing data cleaning, the dataset is examined for missing values across all columns. This helps identify columns that may require further investigation or treatment during the data-cleaning stage.

In [ ]:
# Check missing values in each column

missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)

In [ ]:
# Calculate the percentage of missing values

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": missing_percentage
})

missing_summary = missing_summary[missing_summary["Missing Values"] > 0]

missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

In [ ]:
# Display the number of unique values in each column

df.nunique().sort_values()

## 3.2 Investigating Missing Data

The initial inspection shows that several variables contain missing values. Before deciding how to handle them, the distribution and pattern of missing observations are examined to determine whether the missing values are concentrated in particular records or variables.

In [ ]:
# Count how many columns are missing for each customer

missing_per_row = df.isnull().sum(axis=1)

missing_per_row.value_counts().sort_index()

In [ ]:
# Check how many customers have at least one missing value

customers_with_missing = (missing_per_row > 0).sum()

print("Customers with at least one missing value:", customers_with_missing)
print("Percentage of customers affected:",
      round(customers_with_missing / len(df) * 100, 2), "%")

In [ ]:
# Check whether missing values occur together

missing_matrix = df.isnull().astype(int)

missing_matrix.corr()

In [ ]:
# Display all column names and their data types

column_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Unique Values": df.nunique().values,
    "Missing Values": df.isnull().sum().values
})

column_summary

## 3.3 Duplicate and Missing-Pattern Checks

Before beginning data cleaning, duplicate records and the structure of rows containing missing values are examined. This helps determine whether records can be safely removed or whether missing values should be handled individually.

In [ ]:
# Check for duplicate records

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
# Examine rows containing missing values

rows_with_missing = df[missing_per_row > 0]

rows_with_missing.head(10)

In [ ]:
# Count how many missing values occur in each affected row

missing_per_row.value_counts().sort_index()

In [ ]:
# Check for blank or whitespace-only values in string columns

string_columns = df.select_dtypes(include="object").columns

blank_counts = {}

for column in string_columns:
    blank_counts[column] = df[column].astype(str).str.strip().eq("").sum()

pd.Series(blank_counts).sort_values(ascending=False)

# 4. Data Cleaning & Preparation

The dataset contains missing values, blank entries and some columns with inappropriate data types. The cleaning process is performed carefully to preserve as much useful customer information as possible.

The main cleaning steps are:

1. Convert `TotalCharges` from string to numeric.
2. Treat blank `TotalCharges` values as missing.
3. Handle missing numerical values using median imputation.
4. Handle missing categorical values using an explicit `Unknown` category.
5. Confirm that duplicate records are absent.
6. Verify the cleaned dataset after these operations.

In [ ]:
# Create a copy of the original dataset for cleaning

df_clean = df.copy()

# Convert TotalCharges to numeric.
# Invalid or blank values will become NaN.

df_clean["TotalCharges"] = pd.to_numeric(
    df_clean["TotalCharges"],
    errors="coerce"
)

print("TotalCharges data type after conversion:")
print(df_clean["TotalCharges"].dtype)

print("\nMissing values after conversion:")
print(df_clean.isnull().sum().sort_values(ascending=False))

In [ ]:
# Identify numerical and categorical columns

numerical_columns = df_clean.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = df_clean.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

In [ ]:
# Handle missing numerical values using the median

for column in numerical_columns:
    if df_clean[column].isnull().sum() > 0:
        df_clean[column] = df_clean[column].fillna(
            df_clean[column].median()
        )

# Handle missing categorical values using "Unknown"

for column in categorical_columns:
    if df_clean[column].isnull().sum() > 0:
        df_clean[column] = df_clean[column].fillna("Unknown")

In [ ]:
# Verify missing values after cleaning

remaining_missing = df_clean.isnull().sum()

print("Remaining missing values:")
print(remaining_missing[remaining_missing > 0])

print("\nDataset shape after cleaning:", df_clean.shape)

# 5. Exploratory Data Analysis (EDA)

The cleaned dataset is analysed to identify patterns in customer behaviour and determine how churn varies across important customer characteristics.

The analysis focuses on:

- Overall churn distribution
- Customer tenure
- Monthly charges
- Contract type
- Internet service type
- Customer demographics
- Payment method

# 6. Data Visualisation

The following visualisations are used to identify relationships between customer characteristics and churn. Each visualisation is followed by a business-focused interpretation.

In [ ]:
# Set the visualisation style

sns.set_theme(style="whitegrid")

# Create a figure with 6 visualisations

fig, axes = plt.subplots(2, 3, figsize=(18, 11))


# 1. Churn Distribution

sns.countplot(
    data=df_clean,
    x="Churn",
    ax=axes[0, 0]
)

axes[0, 0].set_title("Customer Churn Distribution")
axes[0, 0].set_xlabel("Churn Status")
axes[0, 0].set_ylabel("Number of Customers")


# 2. Churn by Contract Type

sns.countplot(
    data=df_clean,
    x="Contract",
    hue="Churn",
    ax=axes[0, 1]
)

axes[0, 1].set_title("Churn by Contract Type")
axes[0, 1].set_xlabel("Contract Type")
axes[0, 1].set_ylabel("Number of Customers")
axes[0, 1].tick_params(axis="x", rotation=20)


# 3. Tenure Distribution by Churn

sns.boxplot(
    data=df_clean,
    x="Churn",
    y="tenure",
    ax=axes[0, 2]
)

axes[0, 2].set_title("Tenure Distribution by Churn")
axes[0, 2].set_xlabel("Churn Status")
axes[0, 2].set_ylabel("Tenure (Months)")


# 4. Monthly Charges by Churn

sns.boxplot(
    data=df_clean,
    x="Churn",
    y="MonthlyCharges",
    ax=axes[1, 0]
)

axes[1, 0].set_title("Monthly Charges by Churn")
axes[1, 0].set_xlabel("Churn Status")
axes[1, 0].set_ylabel("Monthly Charges")


# 5. Churn by Internet Service

sns.countplot(
    data=df_clean,
    x="InternetService",
    hue="Churn",
    ax=axes[1, 1]
)

axes[1, 1].set_title("Churn by Internet Service")
axes[1, 1].set_xlabel("Internet Service")
axes[1, 1].set_ylabel("Number of Customers")


# 6. Churn by Payment Method

sns.countplot(
    data=df_clean,
    x="PaymentMethod",
    hue="Churn",
    ax=axes[1, 2]
)

axes[1, 2].set_title("Churn by Payment Method")
axes[1, 2].set_xlabel("Payment Method")
axes[1, 2].set_ylabel("Number of Customers")
axes[1, 2].tick_params(axis="x", rotation=35)


plt.tight_layout()
plt.show()

In [ ]:
# Churn counts and percentages

churn_summary = df_clean["Churn"].value_counts()

churn_percentage = (
    df_clean["Churn"].value_counts(normalize=True) * 100
).round(2)

churn_table = pd.DataFrame({
    "Customer Count": churn_summary,
    "Percentage": churn_percentage
})

churn_table

In [ ]:
# Average numerical features by churn status

numerical_eda = df_clean.groupby("Churn")[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].mean().round(2)

numerical_eda

In [ ]:
# Churn rate by contract type

contract_churn = pd.crosstab(
    df_clean["Contract"],
    df_clean["Churn"],
    normalize="index"
) * 100

contract_churn.round(2)

In [ ]:
# Churn rate by internet service

internet_churn = pd.crosstab(
    df_clean["InternetService"],
    df_clean["Churn"],
    normalize="index"
) * 100

internet_churn.round(2)

# 7. Statistical Analysis

Statistical analysis is performed on the key numerical variables to understand their central tendency and dispersion.

The analysis includes:

- Mean and median to measure central tendency.
- Standard deviation, minimum and maximum values to measure dispersion.
- Quartiles and Interquartile Range (IQR) to identify potential outliers.

In [ ]:
# Statistical summary of key numerical variables

statistical_summary = df_clean[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].describe().T

statistical_summary

In [ ]:
# IQR and outlier analysis

numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]

iqr_results = []

for column in numerical_features:
    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    ]

    iqr_results.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers),
        "Outlier Percentage": round(
            len(outliers) / len(df_clean) * 100, 2
        )
    })

iqr_summary = pd.DataFrame(iqr_results)

iqr_summary

In [ ]:
# Compare mean and median

central_tendency = df_clean[
    ["tenure", "MonthlyCharges", "TotalCharges"]
].agg(["mean", "median"]).T

central_tendency

# 8. Data Preprocessing

Machine learning algorithms require numerical input. The cleaned dataset contains both numerical and categorical variables, so categorical features must be encoded before model training.

The `customerID` column is excluded because it is an identifier and does not provide meaningful predictive information.

The `Churn` column is separated as the target variable.

Categorical predictor variables are converted into numerical features using one-hot encoding.

In [ ]:
# Separate features and target

X = df_clean.drop(columns=["Churn", "customerID"])
y = df_clean["Churn"].map({"No": 0, "Yes": 1})

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# Identify numerical and categorical features

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")

# 9. Train-Test Split

The dataset is divided into training and testing subsets. The training data is used to learn the relationships between customer characteristics and churn, while the test data is reserved for evaluating the model on unseen customers.

A stratified split is used to preserve the proportion of churned and non-churned customers in both datasets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining churn rate:", round(y_train.mean() * 100, 2), "%")
print("Testing churn rate:", round(y_test.mean() * 100, 2), "%")

# 10. Logistic Regression Model

Logistic Regression is used as the primary classification model because the target variable, `Churn`, has two possible outcomes: customers who churn and customers who do not churn.

The model estimates the probability that a customer will churn based on the available customer characteristics.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# Train the model

logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

# 11. Model Evaluation

The Logistic Regression model is evaluated on the unseen test dataset using accuracy, precision, recall, F1-score and a confusion matrix.

These metrics provide a more complete view of model performance than accuracy alone, particularly because identifying customers who are likely to churn is an important business objective.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# Generate predictions

y_pred = logistic_model.predict(X_test)

# Calculate evaluation metrics

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Logistic Regression Performance")
print("-" * 35)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
).plot()

plt.title("Logistic Regression Confusion Matrix")
plt.show()

## 11.1 Model Performance Interpretation

The Logistic Regression model achieved an accuracy of approximately 79.06% on the test dataset.

For the churn class, the model achieved a precision of 62.95%, meaning that when the model predicts a customer will churn, the prediction is correct in a substantial proportion of cases.

The recall for churn was 51.34%. This indicates that the model successfully identifies approximately half of the customers who actually churned. The remaining churned customers were classified as non-churned.

The F1-score for the churn class was 56.55%, showing a moderate balance between precision and recall.

From a business perspective, the model provides a useful baseline for identifying customers who may require retention attention. However, because missing an actual churner can result in a lost retention opportunity, improving churn recall would be an important area for future model improvement.

# 12. Linear Regression Extension

To demonstrate the difference between classification and regression, a Linear Regression model is developed to predict the continuous variable `TotalCharges`.

Unlike Logistic Regression, which predicts a categorical outcome such as customer churn, Linear Regression predicts a continuous numerical value.

In this extension, the model estimates a customer's total charges based on their tenure, monthly charges, services, contract type and other customer characteristics.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create regression features and target

X_reg = df_clean.drop(columns=["TotalCharges", "customerID"])
y_reg = df_clean["TotalCharges"]

# Identify numerical and categorical features

reg_numerical_features = X_reg.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

reg_categorical_features = X_reg.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Numerical features:", reg_numerical_features)
print("Categorical features:", reg_categorical_features)

In [ ]:
# Split regression data

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_reg_train))
print("Testing samples:", len(X_reg_test))

In [ ]:
# Create regression preprocessing pipeline

reg_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            reg_numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            reg_categorical_features
        )
    ]
)

linear_model = Pipeline(
    steps=[
        ("preprocessor", reg_preprocessor),
        ("regressor", LinearRegression())
    ]
)

# Train the model

linear_model.fit(X_reg_train, y_reg_train)

print("Linear Regression model trained successfully.")

In [ ]:
# Generate predictions

y_reg_pred = linear_model.predict(X_reg_test)

# Evaluate the model

mae = mean_absolute_error(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_reg_test, y_reg_pred)

print("Linear Regression Performance")
print("-" * 35)
print(f"MAE : {mae:.2f}")
print(f"MSE : {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

In [ ]:
# Actual vs predicted TotalCharges

plt.figure(figsize=(8, 6))

plt.scatter(
    y_reg_test,
    y_reg_pred,
    alpha=0.5
)

# Perfect prediction reference line

min_value = min(y_reg_test.min(), y_reg_pred.min())
max_value = max(y_reg_test.max(), y_reg_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Total Charges")
plt.ylabel("Predicted Total Charges")
plt.title("Actual vs Predicted Total Charges")

plt.tight_layout()
plt.show()

## 12.1 Linear Regression Interpretation

The Linear Regression model achieved an R² score of 0.5341, indicating that approximately 53.4% of the variation in customer TotalCharges is explained by the features included in the model.

The model achieved a Mean Absolute Error (MAE) of 1118.64 and a Root Mean Squared Error (RMSE) of 1556.34. These results indicate that prediction errors remain substantial, but the model captures a meaningful portion of the relationship between customer characteristics and total charges.

This regression task differs from the churn classification task because Linear Regression predicts a continuous numerical value, whereas Logistic Regression predicts a categorical outcome.

# 13. Feature Interpretation

The coefficients of the Logistic Regression model are examined to identify which customer characteristics have the strongest association with the predicted probability of churn.

Positive coefficients indicate an increased tendency toward the churn class, while negative coefficients indicate a decreased tendency toward churn, with other model features held constant.

In [ ]:
# Extract feature names from the trained preprocessing pipeline

preprocessor_fitted = logistic_model.named_steps["preprocessor"]
classifier = logistic_model.named_steps["classifier"]

feature_names = preprocessor_fitted.get_feature_names_out()

coefficients = classifier.coef_[0]

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

# Add absolute coefficient magnitude

feature_importance["Absolute Coefficient"] = (
    feature_importance["Coefficient"].abs()
)

# Display strongest features

feature_importance = feature_importance.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

feature_importance.head(15)

In [ ]:
# Visualise the 10 strongest Logistic Regression coefficients

top_features = feature_importance.head(10).sort_values(
    by="Coefficient"
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_features["Feature"],
    top_features["Coefficient"]
)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Top Features Associated with Customer Churn")

plt.tight_layout()
plt.show()

## 13.1 Feature Interpretation

The Logistic Regression coefficients provide insight into which customer characteristics are most strongly associated with churn predictions.

The strongest positive coefficient was associated with Fiber optic internet service, suggesting a higher predicted tendency toward churn compared with the reference category. Electronic check payment and paperless billing also showed positive associations with churn.

Contract type showed some of the strongest negative coefficients. One-year and especially two-year contracts were associated with a substantially lower predicted tendency toward churn compared with the reference contract category.

Online Security and Phone Service also showed negative associations with churn.

These coefficients represent associations within the fitted model and should not be interpreted as proof of causation.

# 14. Client Risk Insights

The trained Logistic Regression model produces a churn probability for each customer in the test dataset. These probabilities are converted into practical risk categories to help the client prioritise retention efforts.

Risk categories are defined as:

- Low Risk: churn probability below 30%
- Medium Risk: churn probability from 30% to below 60%
- High Risk: churn probability of 60% or higher

In [ ]:
# Generate churn probabilities

churn_probability = logistic_model.predict_proba(X_test)[:, 1]

risk_insights = X_test.copy()

risk_insights["Actual_Churn"] = y_test.values
risk_insights["Churn_Probability"] = churn_probability

# Assign risk categories

risk_insights["Risk_Category"] = pd.cut(
    risk_insights["Churn_Probability"],
    bins=[-0.01, 0.30, 0.60, 1.00],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# Display risk distribution

risk_distribution = (
    risk_insights["Risk_Category"]
    .value_counts()
    .sort_index()
)

risk_distribution

In [ ]:
# Risk category percentages

risk_percentage = (
    risk_insights["Risk_Category"]
    .value_counts(normalize=True)
    .sort_index() * 100
).round(2)

risk_summary = pd.DataFrame({
    "Customer Count": risk_distribution,
    "Percentage": risk_percentage
})

risk_summary

In [ ]:
# Churn rate within each risk category

risk_churn_rate = (
    risk_insights.groupby("Risk_Category", observed=True)["Actual_Churn"]
    .mean() * 100
)

risk_churn_rate.round(2)

# 15. Business Recommendations

Based on the exploratory analysis, Logistic Regression results, feature coefficients, and customer risk segmentation, the following business recommendations are proposed.

### 1. Prioritise High-Risk Customers

High-risk customers should receive the highest priority for retention activities. This group represents 14.12% of the test customers and has an observed churn rate of 69.85%.

The company can consider targeted retention offers, proactive customer support, contract incentives, or personalised communication for this segment.

### 2. Focus on Month-to-Month Customers

Month-to-month contracts show substantially higher churn compared with one-year and two-year contracts. The company should consider incentives that encourage customers to move toward longer-term contracts.

Possible approaches include discounts, loyalty benefits, or additional services for customers willing to commit to longer contracts.

### 3. Investigate Fiber Optic Customer Churn

Fiber optic customers show a strong positive association with churn in the Logistic Regression model. The company should investigate service quality, pricing, customer support, and competitor offerings for this customer segment.

### 4. Target Electronic Check Customers

Electronic check payment has a positive association with churn in the model. Customers using this payment method could be included in targeted retention campaigns and encouraged to adopt convenient automatic payment methods where appropriate.

### 5. Strengthen Online Security and Technical Support

Online Security and Tech Support show negative associations with churn. Promoting these services and improving customer awareness of their benefits may support customer retention.

### 6. Use Risk-Based Retention

Rather than applying the same retention strategy to every customer, the company should prioritise customers according to predicted churn risk.

High-risk customers should receive the most intensive intervention, medium-risk customers should receive preventive engagement, and low-risk customers can be maintained through regular service and loyalty activities.

# 16. Conclusion

This project developed an end-to-end customer churn prediction and business insight system using the Telco Customer Churn dataset.

The analysis included data loading, data quality assessment, missing-value treatment, exploratory data analysis, statistical analysis, feature preprocessing, Logistic Regression classification, Linear Regression, and customer risk segmentation.

The Logistic Regression model achieved an accuracy of 79.06%, with a churn recall of 51.34% and an F1-score of 56.55%. The model provides a useful baseline for identifying customers who may be at risk of churn.

The risk segmentation further demonstrated practical value. The observed churn rate increased from 11.14% for low-risk customers to 39.94% for medium-risk customers and 69.85% for high-risk customers.

The analysis indicates that contract type, internet service, payment method, and additional customer services are among the important factors associated with churn. These findings can support targeted retention strategies and help the company allocate retention resources more effectively.

Overall, the project demonstrates how data analysis and machine learning can be combined to transform customer data into actionable business insights.

In [ ]:
# Save final project visualizations

import os

os.makedirs("../visualizations", exist_ok=True)

# 1. Customer Churn Distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig("../visualizations/01_customer_churn_distribution.png", dpi=300)
plt.show()


# 2. Churn by Contract Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="Contract", hue="Churn")
plt.title("Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("../visualizations/02_churn_by_contract.png", dpi=300)
plt.show()


# 3. Tenure Distribution by Churn
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x="Churn", y="tenure")
plt.title("Tenure Distribution by Churn")
plt.xlabel("Churn Status")
plt.ylabel("Tenure (Months)")
plt.tight_layout()
plt.savefig("../visualizations/03_tenure_by_churn.png", dpi=300)
plt.show()


# 4. Monthly Charges by Churn
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn")
plt.xlabel("Churn Status")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.savefig("../visualizations/04_monthly_charges_by_churn.png", dpi=300)
plt.show()


# 5. Churn by Internet Service
plt.figure(figsize=(8, 5))
sns.countplot(data=df_clean, x="InternetService", hue="Churn")
plt.title("Churn by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig("../visualizations/05_churn_by_internet_service.png", dpi=300)
plt.show()


# 6. Churn by Payment Method
plt.figure(figsize=(9, 5))
sns.countplot(data=df_clean, x="PaymentMethod", hue="Churn")
plt.title("Churn by Payment Method")
plt.xlabel("Payment Method")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("../visualizations/06_churn_by_payment_method.png", dpi=300)
plt.show()

print("All visualizations saved successfully!")